# 使用 Raha 和 Baran 的端到端数据清洗流水线（最小化顺序执行版）
我们使用无需配置的错误检测和错误纠正系统 Raha 与 Baran，构建一条端到端数据清洗流水线。

In [30]:
import pandas
import IPython.display

import raha

## 使用 Raha 进行错误检测

### 1. 实例化 Detection 类
首先实例化 `Detection` 类。

In [4]:
app_1 = raha.Detection()

# 你想标注多少个元组？
app_1.LABELING_BUDGET = 20

# 是否查看日志？
app_1.VERBOSE = True

### 2. 实例化数据集
接着加载并实例化数据集对象。

In [10]:
dataset_dictionary = {
    "name": "flights",
    "path": "../datasets/flights/dirty.csv",
    "clean_path": "../datasets/flights/clean.csv"
}
d = app_1.initialize_dataset(dataset_dictionary)
d.dataframe.head()

,tuple_id,src,flight,sched_dep_time,act_dep_time,sched_arr_time,act_arr_time
0,1,aa,AA-3859-IAH-ORD,7:10 a.m.,7:16 a.m.,9:40 a.m.,9:32 a.m.
1,2,aa,AA-1733-ORD-PHX,7:45 p.m.,7:58 p.m.,10:30 p.m.,
2,3,aa,AA-1640-MIA-MCO,6:30 p.m.,,7:25 p.m.,
3,4,aa,AA-518-MIA-JFK,6:40 a.m.,6:54 a.m.,9:25 a.m.,9:28 a.m.
4,5,aa,AA-3756-ORD-SLC,12:15 p.m.,12:41 p.m.,2:45 p.m.,2:50 p.m.


### 3. 运行错误检测策略
Raha 会在数据集上运行全部或有潜力的错误检测策略。由于需要在数据集上运行所有策略，此步骤可能需要一些时间。

In [31]:
app_1.run_strategies(d)

167 strategy profiles are collected.


I just load strategies' results as they have already been run on the dataset!


### 4. 生成特征
随后，Raha 会基于错误检测策略的输出，为每个数据单元生成特征向量。

In [32]:
app_1.generate_features(d)

13 Features are generated for column 0.
20 Features are generated for column 1.
24 Features are generated for column 2.
28 Features are generated for column 3.
35 Features are generated for column 4.
33 Features are generated for column 5.
35 Features are generated for column 6.


### 5. 构建聚类
接下来，Raha 会为基于聚类的采样方法构建层次聚类模型。

In [33]:
app_1.build_clusters(d)

A hierarchical clustering model is built for column 0.
A hierarchical clustering model is built for column 1.
A hierarchical clustering model is built for column 2.
A hierarchical clustering model is built for column 3.
A hierarchical clustering model is built for column 4.
A hierarchical clustering model is built for column 5.
A hierarchical clustering model is built for column 6.


### 6. 交互式元组采样与标注
随后，Raha 会迭代采样元组。我们需要标注每个采样元组中的数据单元。

In [35]:
while len(d.labeled_tuples) < app_1.LABELING_BUDGET:
    app_1.sample_tuple(d)
    if d.has_ground_truth:
        app_1.label_with_ground_truth(d)
    else:
        print("请标注以下采样元组中的脏数据单元。")
        sampled_tuple = pandas.DataFrame(data=[d.dataframe.iloc[d.sampled_tuple, :]], columns=d.dataframe.columns)
        IPython.display.display(sampled_tuple)
        for j in range(d.dataframe.shape[1]):
            cell = (d.sampled_tuple, j)
            value = d.dataframe.iloc[cell]
            correction = input("What is the correction for value '{}'? Type in the same value if it is not erronous.\n".format(value))
            user_label = 1 if value != correction else 0
            d.labeled_cells[cell] = [user_label, correction]
        d.labeled_tuples[d.sampled_tuple] = 1

### 7. 传播用户标签
随后，Raha 会通过所属聚类传播每个用户标签。

In [36]:
app_1.propagate_labels(d)

The number of labeled data cells increased from 140 to 8117.


### 8. 预测数据单元标签
随后，Raha 会为每个数据列训练并应用一个分类器，用于预测其余数据单元的标签。

In [37]:
app_1.predict_labels(d)

A classifier is trained and applied on column 0.
A classifier is trained and applied on column 1.
A classifier is trained and applied on column 2.
A classifier is trained and applied on column 3.
A classifier is trained and applied on column 4.
A classifier is trained and applied on column 5.
A classifier is trained and applied on column 6.


### 9. 存储结果
Raha 也可以存储错误检测结果。

In [38]:
app_1.store_results(d)

The results are stored in ../datasets/flights\raha-baran-results-flights\error-detection\detection.dataset.


### 10. 评估错误检测任务
最后，我们可以评估错误检测任务。

In [39]:
p, r, f = d.get_data_cleaning_evaluation(d.detected_cells)[:3]
print("Raha 在 {} 上的性能：\n精确率 = {:.2f}\n召回率 = {:.2f}\nF1 = {:.2f}".format(d.name, p, r, f))

Raha's performance on flights:
Precision = 0.86
Recall = 0.74
F1 = 0.80


# 使用 Baran 进行错误纠正

### 1. 实例化 Correction 类
首先实例化 `Correction` 类。

In [40]:
app_2 = raha.Correction()

# 你想标注多少个元组？
app_2.LABELING_BUDGET = 20

# 是否查看日志？
app_2.VERBOSE = True

### 2. 初始化数据集对象
接着初始化数据集对象。

In [41]:
d = app_2.initialize_dataset(d)
d.dataframe.head()

,tuple_id,src,flight,sched_dep_time,act_dep_time,sched_arr_time,act_arr_time
0,1,aa,AA-3859-IAH-ORD,7:10 a.m.,7:16 a.m.,9:40 a.m.,9:32 a.m.
1,2,aa,AA-1733-ORD-PHX,7:45 p.m.,7:58 p.m.,10:30 p.m.,
2,3,aa,AA-1640-MIA-MCO,6:30 p.m.,,7:25 p.m.,
3,4,aa,AA-518-MIA-JFK,6:40 a.m.,6:54 a.m.,9:25 a.m.,9:28 a.m.
4,5,aa,AA-3756-ORD-SLC,12:15 p.m.,12:41 p.m.,2:45 p.m.,2:50 p.m.


### 3. 初始化错误纠正模型
Baran 会初始化错误纠正模型。

In [42]:
app_2.initialize_models(d)

The error corrector models are initialized.


### 4. 交互式元组采样、标注、模型更新、特征生成与纠正预测
随后，Baran 会迭代采样元组。我们需要标注每个采样元组中的数据单元。之后，它会相应地更新模型，并为每一对数据错误和候选纠正值生成特征向量。最后，它会为每个数据列训练并应用一个分类器，以预测每个数据错误的最终纠正值。由于我们已经为 Raha 标注过元组，这里会复用相同的已标注元组，不再标注新的元组。

In [43]:
# while len(d.labeled_tuples) < app_2.LABELING_BUDGET:
#     app_2.sample_tuple(d)
#     if d.has_ground_truth:
#         app_2.label_with_ground_truth(d)
#     else:
#         print("请标注以下采样元组中的脏数据单元。")
#         sampled_tuple = pandas.DataFrame(data=[d.dataframe.iloc[d.sampled_tuple, :]], columns=d.dataframe.columns)
#         IPython.display.display(sampled_tuple)
#         for j in range(d.dataframe.shape[1]):
#             cell = (d.sampled_tuple, j)
#             value = d.dataframe.iloc[cell]
#             correction = input("值 '{}' 的纠正值是什么？如果该值没有错误，请输入相同的值。\n".format(value))
#             user_label = 1 if value != correction else 0
#             d.labeled_cells[cell] = [user_label, correction]
#         d.labeled_tuples[d.sampled_tuple] = 1
#     app_2.update_models(d)
#     app_2.predict_corrections(d)

for si in d.labeled_tuples:
    d.sampled_tuple = si
    app_2.update_models(d)
    app_2.predict_corrections(d)

The error corrector models are updated with new labeled tuple 1030.
Predicting module...
------------------------------------------------------------------------
1/4 columns(sched_dep_time)
Generating train features(9) ...
846 pairs of (a data error, a potential correction) are featurized.
Training classifier ...
Predicting corrections...
------------------------------------------------------------------------
2/4 columns(act_dep_time)
Generating train features(16) ...
930 pairs of (a data error, a potential correction) are featurized.
Training classifier ...
Predicting corrections...
------------------------------------------------------------------------
3/4 columns(sched_arr_time)
Generating train features(9) ...
1548 pairs of (a data error, a potential correction) are featurized.
Training classifier ...
Predicting corrections...
------------------------------------------------------------------------
4/4 columns(act_arr_time)
Generating train features(10) ...
2482 pairs of (a data 

### 5. 存储结果
Baran 也可以存储错误纠正结果。

In [17]:
app_2.store_results(d)

The results are stored in ../datasets/flights/raha-baran-results-flights/error-correction/correction.dataset.


### 6. 评估错误纠正任务
最后，我们可以评估错误纠正任务。

In [18]:
p, r, f = d.get_data_cleaning_evaluation(d.corrected_cells)[-3:]
print("Baran 在 {} 上的性能：\n精确率 = {:.2f}\n召回率 = {:.2f}\nF1 = {:.2f}".format(d.name, p, r, f))

Baran's performance on flights:
Precision = 0.88
Recall = 0.53
F1 = 0.66
